# v15 — token balance check

Answers one question: **does the encoder token actually encode the weather, or only the
station metadata and the calendar?**

Run on Renku, where `torch` and the checkpoints live. Reports:

1. **Did training escape the initialisation?** — trained `var_weights` / `var_type_embedding`
   scales vs their init values. This is the decisive check.
2. **Component decomposition** — RMS and variance-across-tokens of the five summed embeddings.
3. **Content fraction** — `Var(obs)/Var(total)`, across tokens, so constants score zero.
4. **Permutation battery** — how far the token moves when the weather / station metadata /
   position / time are permuted. Weather should move it as much as the others do.
5. **What the proposed fix would give** — the same metrics with the init repaired.

Nothing is written to disk; the checkpoint is opened read-only.

## 0 · Locate the project

In [1]:
from pathlib import Path

# Leave as None to auto-locate. Set manually if the search picks the wrong one.
PROJECT_SRC   = None     # dir containing model/embeddings.py  (".../Station MAE/src")
CHECKPOINT    = None     # ".../checkpoints/full_run_cloud_v15/best.ckpt"
DATA_ROOT     = None     # ".../PeakWeatherDataset"

USE_REAL_DATA = True
B, W, N_MAX   = 2, 72, 155
SEED          = 0

def _find_src():
    here, seen = Path.cwd().resolve(), []
    for base in [here, *here.parents][:7]:
        for cand in [base / "src", base, *sorted(base.glob("*/src"))]:
            seen.append(cand)
            if (cand / "model" / "embeddings.py").is_file():
                return cand, seen
    return None, seen

if PROJECT_SRC is None:
    PROJECT_SRC, _searched = _find_src()

if PROJECT_SRC is None:
    print("!! Could not locate src/. Searched:")
    for p in dict.fromkeys(_searched):
        print("     ", p)
    print("\n   Set PROJECT_SRC above to the folder containing model/embeddings.py")
else:
    PROJECT_SRC = Path(PROJECT_SRC)
    _root = PROJECT_SRC.parent
    print(f"PROJECT_SRC : {PROJECT_SRC}")

    if CHECKPOINT is None:
        _skip = ("lstm", "simple-mae", "simple_mae", "masked-tf", "masked_transformer")
        _ok   = lambda p: not any(k in str(p).lower() for k in _skip)
        hits  = ([p for p in sorted(_root.glob("checkpoints/*v15*/best.ckpt"))]
                 or [p for p in sorted(_root.glob("checkpoints/full_run*/best.ckpt"))]
                 or [p for p in sorted(_root.glob("checkpoints/*v15*/last.ckpt"))]
                 or [p for p in sorted(_root.glob("checkpoints/*/best.ckpt")) if _ok(p)])
        CHECKPOINT = hits[0] if hits else None
        if hits:
            print(f"CHECKPOINT  : {CHECKPOINT}")
            for h in hits[1:6]:
                print(f"              also found: {h}")
        else:
            print("CHECKPOINT  : none found -> init-only analysis")

    if DATA_ROOT is None:
        d = sorted(_root.parent.glob("**/PeakWeatherDataset"))
        if not d and Path("/home/renku/work").exists():
            d = sorted(Path("/home/renku/work").glob("*PeakWeather*"))
        DATA_ROOT = str(d[0]) if d else None
        print(f"DATA_ROOT   : {DATA_ROOT or 'not found -> synthetic observations'}")

PROJECT_SRC : /Users/aureliedejong/PycharmProjects/weather-station-model-with-transformers/Station MAE/src
CHECKPOINT  : /Users/aureliedejong/PycharmProjects/weather-station-model-with-transformers/Station MAE/checkpoints/full_run_cloud_v15/best.ckpt
DATA_ROOT   : /Users/aureliedejong/PycharmProjects/weather-station-model-with-transformers/Station MAE/PeakWeatherDataset


## 1 · Load

In [2]:
import sys, math, torch, numpy as np

if PROJECT_SRC is None:
    raise SystemExit("PROJECT_SRC is not set — fix section 0 first.")
sys.path.insert(0, str(PROJECT_SRC))
torch.manual_seed(SEED); np.random.seed(SEED)

from model.embeddings import (
    VariableProjection, PositionalEmbedding, StationEmbedding,
    TemporalEmbedding, StepIndexEmbedding,
    POSITION_FOURIER_DIM, TEMPORAL_FOURIER_DIM, STATION_CHAR_DIM, NUM_VARIABLES,
)
print("imports OK")

ckpt = sd = cfg = None
PREFIX = "model."            # defined up-front: the guards below rely on it

if CHECKPOINT and Path(CHECKPOINT).is_file():
    ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    sd   = ckpt["state_dict"]
    cfg  = ckpt.get("hyper_parameters", {}).get("cfg", {}) or {}

    # torch.compile wraps the module -> keys are model._orig_mod.encoder.*
    if any(k.startswith("model._orig_mod.") for k in sd):
        PREFIX = "model._orig_mod."

    print(f"checkpoint : {Path(CHECKPOINT).name}   epoch {ckpt.get('epoch','?')}"
          f"   global_step {ckpt.get('global_step','?')}")
    print(f"d_model {cfg.get('d_model')}  enc_layers {cfg.get('enc_layers')}  "
          f"temporal_patch {cfg.get('temporal_patch')}  mask_ratio {cfg.get('mask_ratio')}")
    print(f"key prefix : {PREFIX}"
          + ("   (trained with --compile)" if "_orig_mod" in PREFIX else ""))

    # Guard: this notebook only understands StationMAE (v15) checkpoints.
    if f"{PREFIX}encoder.var_proj.var_weights" not in sd:
        _pre = sorted({".".join(k.split(".")[:3]) for k in sd})[:8]
        raise SystemExit(
            f"\n!! {Path(CHECKPOINT).name} is not a StationMAE checkpoint.\n"
            f"   key prefixes present: {_pre}\n"
            f"   Set CHECKPOINT in section 0 to a v15 run (or None for init-only).")
    if any("input_cross_attn" in k for k in sd):
        print("!! WARNING: this checkpoint predates v15 (decoder.input_cross_attn.*).")
else:
    print("No checkpoint — init-only analysis with fresh random weights.")

D = (cfg or {}).get("d_model", 384)
V = NUM_VARIABLES
print(f"\nd_model = {D}, num_vars = {V}")

imports OK
checkpoint : best.ckpt   epoch 61   global_step 38750
d_model 384  enc_layers 8  temporal_patch 3  mask_ratio 0.5
key prefix : model._orig_mod.   (trained with --compile)

d_model = 384, num_vars = 6


## 2 · Did training escape the initialisation?

`var_weights` is initialised with `xavier_uniform_` on shape `(6, d_model)` → bound
`sqrt(6/(6+d))` = 0.124, std ≈ 0.072. The `Linear(1, d_model)` modules it replaced had
`fan_in = 1` → bound 1.0, std ≈ 0.577 — eight times larger for the same function.
`var_type_embedding` uses PyTorch's default `N(0,1)`, whereas every other learned
token-level parameter in this project uses `trunc_normal_(std=0.02)`.

If the trained values sit near their init, the model never left the metadata-dominated
basin. If `var_weights` has grown an order of magnitude, it escaped on its own.

In [3]:
init_vw = math.sqrt(6.0/(V+D))/math.sqrt(3)     # = 0.0716 at d=384
init_vt = 1.0

print(f"{'parameter':34s} {'init std':>10s} {'trained std':>12s} {'ratio':>8s}")
print("-"*68)
if sd is not None:
    for key, init in [("var_weights", init_vw),
                      ("var_type_embedding.weight", init_vt),
                      ("var_absent_embedding", 0.0)]:
        full = f"{PREFIX}encoder.var_proj.{key}"
        if full not in sd:
            print(f"  {key:32s}  NOT FOUND in state_dict")
            continue
        s_ = sd[full].float().std().item()
        r  = f"{s_/init:.1f}x" if init > 0 else "n/a"
        print(f"  {key:32s} {init:10.4f} {s_:12.4f} {r:>8s}")
    vw = sd.get(f"{PREFIX}encoder.var_proj.var_weights")
    if vw is not None:
        ratio = vw.float().std().item()/init_vw
        print()
        if   ratio < 2: print(f"VERDICT  var_weights grew only {ratio:.1f}x — training did NOT escape.")
        elif ratio < 8: print(f"VERDICT  var_weights grew {ratio:.1f}x — partial escape.")
        else:           print(f"VERDICT  var_weights grew {ratio:.1f}x — escaped on its own.")
else:
    print("  (no checkpoint — skipping)")

parameter                            init std  trained std    ratio
--------------------------------------------------------------------
  var_weights                          0.0716       0.2757     3.9x
  var_type_embedding.weight            1.0000       1.0164     1.0x
  var_absent_embedding                 0.0000       0.0251      n/a

VERDICT  var_weights grew 3.9x — partial escape.


## 3 · Rebuild the embedding stack

In [4]:
def build(trained: bool):
    mods = {
        "var_proj":     VariableProjection(num_vars=V, d_model=D),
        "pos_emb":      PositionalEmbedding(d_model=D, fourier_dim=POSITION_FOURIER_DIM),
        "station_emb":  StationEmbedding(d_model=D, input_dim=STATION_CHAR_DIM),
        "temporal_emb": TemporalEmbedding(d_model=D, fourier_dim=TEMPORAL_FOURIER_DIM),
        "step_emb":     StepIndexEmbedding(d_model=D),
    }
    token_norm = torch.nn.LayerNorm(D)
    if trained and sd is not None:
        for name, m in mods.items():
            pre = f"{PREFIX}encoder.{name}."
            sub = {k[len(pre):]: v for k, v in sd.items() if k.startswith(pre)}
            missing, unexpected = m.load_state_dict(sub, strict=False)
            # LOUD on purpose: a silent strict=False caused the v9-v13 eval bug.
            if missing or unexpected:
                print(f"  !! {name}: missing={list(missing)} unexpected={list(unexpected)}")
            else:
                print(f"  ok {name}: {len(sub)} tensors")
        pre = f"{PREFIX}encoder.token_norm."
        sub = {k[len(pre):]: v for k, v in sd.items() if k.startswith(pre)}
        if sub:
            token_norm.load_state_dict(sub)
    for m in mods.values():
        m.eval()
    token_norm.eval()
    return mods, token_norm

print("fresh init:")
FRESH, FRESH_LN = build(trained=False)
if sd is not None:
    print("\ntrained:")
    TRAINED, TRAINED_LN = build(trained=True)
else:
    TRAINED, TRAINED_LN = FRESH, FRESH_LN

fresh init:

trained:
  ok var_proj: 4 tensors
  ok pos_emb: 7 tensors
  ok station_emb: 6 tensors
  ok temporal_emb: 7 tensors
  ok step_emb: 7 tensors


/opt/anaconda3/envs/capstone/lib/python3.12/site-packages/torch/nn/modules/module.py:2589: RuntimeWarning: VariableProjection: migrated a pre-rebalance checkpoint at '' — var_type_embedding folded into var_biases / var_absent_embedding and all three tensors scaled by 1/sqrt(6). The loaded model is numerically IDENTICAL to the one that was trained.
  module._load_from_state_dict(


## 4 · Get a batch

In [5]:
batch = None
if USE_REAL_DATA and DATA_ROOT:
    try:
        from data.dataset import load_peakweather, StationMAEDataset
        ds = load_peakweather(root=DATA_ROOT)
        tr = StationMAEDataset(ds, split="train", window_size=W,
                               delta_steps=36, max_delta_steps=36,
                               cache_dir=DATA_ROOT, shared_memory=False,
                               delta_mode="fixed_grid", delta_grid_stride=3,
                               exclude_stations=["PFA"])
        items = [tr[i] for i in range(B)]
        batch = {k: torch.stack([it[k] for it in items]) for k in ("x", "x_mask", "x_hours")}
        sp = items[0]["spatial"]
        batch["spatial"] = (sp[0] if sp.dim() == 3 else sp).clone()
        print(f"REAL data: x {tuple(batch['x'].shape)}  spatial {tuple(batch['spatial'].shape)}")
    except Exception as e:
        print(f"real data unavailable ({type(e).__name__}: {e})\n-> synthetic")

if batch is None:
    batch = {"x":       torch.randn(B, W, N_MAX, V),
             "x_mask":  (torch.rand(B, W, N_MAX, V) > 0.1).float(),
             "x_hours": torch.rand(B, W)*1e3 + 4e5,
             "spatial": torch.randn(N_MAX, 15)}
    batch["x"] *= batch["x_mask"]
    print(f"SYNTHETIC data: x {tuple(batch['x'].shape)}  (normalised obs ~ N(0,1))")

Nn = batch["x"].shape[2]

[Cache] Loading from /Users/aureliedejong/PycharmProjects/weather-station-model-with-transformers/Station MAE/PeakWeatherDataset/peakweather_obs_cache.pt …
[Cache] Loaded in 0.8s  (obs (461952, 156, 6))
[StationFilter] Excluded 1 station(s) ['PFA']  → N=155 (was 156)
REAL data: x (2, 72, 155, 6)  spatial (155, 15)


## 5 · Component decomposition and content fraction

`Var` is taken **across tokens** and summed over `d_model`, so a component that is identical
for every token contributes zero — which is the point.

In [6]:
def components(mods, b):
    '''Reproduce StationMAEEncoder._build_tokens; each term returned at (B,W,N,d).'''
    x, xm, sp, hrs = b["x"], b["x_mask"], b["spatial"], b["x_hours"]
    Bs, Ws, Ns, Vv = x.shape
    with torch.no_grad():
        obs = mods["var_proj"](x.reshape(Bs*Ws, Ns, Vv),
                               xm.reshape(Bs*Ws, Ns, Vv)).reshape(Bs, Ws, Ns, -1)
        spb = sp.unsqueeze(0) if sp.dim() == 2 else sp
        pos = mods["pos_emb"](spb[..., :2]).unsqueeze(1)
        sta = mods["station_emb"](spb[..., 2:]).unsqueeze(1)
        tem = mods["temporal_emb"](hrs).unsqueeze(2)
        stp = mods["step_emb"](torch.arange(Ws)).view(1, Ws, 1, -1)
    d = obs.shape[-1]
    e = lambda t: t.expand(Bs, Ws, Ns, d)
    return {"obs (CONTENT)": obs, "position p1": e(pos), "station p2": e(sta),
            "temporal t": e(tem), "step s": e(stp)}

def report(mods, label):
    c = components(mods, batch)
    print(f"\n=== {label} ===")
    print(f"{'component':18s} {'RMS/token':>11s} {'var across tokens':>19s}")
    var = {}
    for k, z in c.items():
        var[k] = z.reshape(-1, z.shape[-1]).var(0).sum().item()
        print(f"  {k:18s} {z.pow(2).mean(-1).sqrt().mean():11.4f} {var[k]:19.3f}")
    cf = 100*var["obs (CONTENT)"]/sum(var.values())
    # Parity for FIVE summed branches is 20% each - not 50%. A healthy value
    # is the same order as parity; the broken regime was 0.04%.
    print(f"\n  CONTENT FRACTION = {cf:.2f}%     (parity for 5 branches = 20.00%, floor = 5%)")
    vp = mods["var_proj"]
    with torch.no_grad():
        # token-independent part = var_proj evaluated at zero observations
        const = vp(torch.zeros(1, 1, vp.num_vars), torch.ones(1, 1, vp.num_vars))
    sig = c["obs (CONTENT)"] - const
    print(f"    constant part (var_type + biases) RMS {const.pow(2).mean().sqrt():.4f}")
    print(f"    data-dependent (the weather)      RMS {sig.pow(2).mean().sqrt():.4f}")
    return c, cf

C_fresh, cf_fresh = report(FRESH, "FRESH INIT")
cf_train = None
if sd is not None:
    C_train, cf_train = report(TRAINED, "TRAINED CHECKPOINT")


=== FRESH INIT ===
component            RMS/token   var across tokens
  obs (CONTENT)           0.5243              62.610
  position p1             0.5807             109.609
  station p2              0.5849             116.041
  temporal t              0.5800              58.881
  step s                  0.5762              93.190

  CONTENT FRACTION = 14.22%     (healthy: 30-60%)
    constant part (var_type + biases) RMS 0.0000
    data-dependent (the weather)      RMS 0.5377

=== TRAINED CHECKPOINT ===
component            RMS/token   var across tokens
  obs (CONTENT)           0.4144               2.815
  position p1             0.3489              22.628
  station p2              0.2775              18.800
  temporal t              0.4385              12.437
  step s                  0.4062              42.675

  CONTENT FRACTION = 2.83%     (healthy: 30-60%)
    constant part (var_type + biases) RMS 0.4045
    data-dependent (the weather)      RMS 0.1122


## 6 · Permutation battery

Permute one input factor, leave the rest alone, measure how far the **normalised** token
moves — i.e. what the first transformer block actually sees.

The weather row should be the same order of magnitude as the other three.

In [7]:
def perm_battery(mods, ln, label):
    g = torch.Generator().manual_seed(SEED)
    with torch.no_grad():
        base = ln(sum(components(mods, batch).values()))
    out = {}
    for what in ("weather", "station", "position", "time"):
        b2 = {k: v.clone() for k, v in batch.items()}
        if what == "weather":
            pi = torch.randperm(Nn, generator=g)
            b2["x"], b2["x_mask"] = b2["x"][:, :, pi], b2["x_mask"][:, :, pi]
        elif what == "station":
            pi = torch.randperm(Nn, generator=g); b2["spatial"][:, 2:] = batch["spatial"][pi, 2:]
        elif what == "position":
            pi = torch.randperm(Nn, generator=g); b2["spatial"][:, :2] = batch["spatial"][pi, :2]
        else:
            pi = torch.randperm(W,  generator=g); b2["x_hours"] = b2["x_hours"][:, pi]
        with torch.no_grad():
            pert = ln(sum(components(mods, b2).values()))
        out[what] = ((pert-base).norm()/base.norm()).item()
    print(f"\n=== {label} ===")
    for k, v in out.items():
        print(f"  {k:9s} permutation -> relative token change {v:.4f}"
              + ("   <-- should match the others" if k == "weather" else ""))
    ratio = out["weather"]/float(np.mean([out[k] for k in ("station","position","time")]))
    # Cross-check only, NOT a gate: this ratio is measured post-LayerNorm and
    # its denominator averages three already-large perturbations, so it reads
    # high even when the content fraction is poor. Trust content fraction.
    print(f"\n  weather / metadata ratio = {ratio:.3f}   (parity ~ 1.0)")
    return out, ratio

P_fresh, r_fresh = perm_battery(FRESH, FRESH_LN, "FRESH INIT")
r_train = None
if sd is not None:
    P_train, r_train = perm_battery(TRAINED, TRAINED_LN, "TRAINED CHECKPOINT")


=== FRESH INIT ===
  weather   permutation -> relative token change 0.4166   <-- should match the others
  station   permutation -> relative token change 0.6138
  position  permutation -> relative token change 0.5776
  time      permutation -> relative token change 0.4438

  weather / metadata ratio = 0.764   (healthy: > 0.3)

=== TRAINED CHECKPOINT ===
  weather   permutation -> relative token change 0.1599   <-- should match the others
  station   permutation -> relative token change 0.4390
  position  permutation -> relative token change 0.4763
  time      permutation -> relative token change 0.3668

  weather / metadata ratio = 0.374   (healthy: > 0.3)


## 7 · What the proposed fix would give

Applied to an in-memory copy; nothing is saved.

| | current | proposed |
|---|---|---|
| `var_weights` init | `xavier_uniform_` on `(6,d)` → std 0.072 | `Linear(1,d)` scale → std 0.577 |
| `var_type_embedding` init | `N(0,1)` | `trunc_normal_(std=0.02)` |
| divisor | `/ num_vars` | `/ sqrt(num_vars)` |

In [9]:
import copy

def apply_fix(mods):
    m = {k: copy.deepcopy(v) for k, v in mods.items()}
    vp = m["var_proj"]
    with torch.no_grad():
        vp.var_weights.mul_(0.5773/max(vp.var_weights.std().item(), 1e-8))
        w = vp.var_type_embedding.weight
        w.mul_(0.02/max(w.std().item(), 1e-8))
    _orig = vp.forward
    vp.forward = lambda x, mask: _orig(x, mask)*math.sqrt(vp.num_vars)   # /V -> /sqrt(V)
    return m

BASE_MODS = TRAINED if sd is not None else FRESH
BASE_LN   = TRAINED_LN if sd is not None else FRESH_LN
FIXED     = apply_fix(BASE_MODS)

_, cf_fixed = report(FIXED, "AFTER THE PROPOSED FIX")
_, r_fixed  = perm_battery(FIXED, BASE_LN, "AFTER THE PROPOSED FIX")

AttributeError: 'VariableProjection' object has no attribute 'var_type_embedding'

## 8 · Summary — paste this back

In [10]:
print("="*68)
print("v15 TOKEN BALANCE — SUMMARY")
print("="*68)
if sd is not None:
    print(f"checkpoint       {Path(CHECKPOINT).name}   epoch {ckpt.get('epoch','?')}")
    vw = sd.get(f"{PREFIX}encoder.var_proj.var_weights")
    vt = sd.get(f"{PREFIX}encoder.var_proj.var_type_embedding.weight")
    if vw is not None:
        print(f"var_weights std  {vw.float().std().item():.4f}   "
              f"(init {init_vw:.4f} -> {vw.float().std().item()/init_vw:.1f}x)")
    if vt is not None:
        print(f"var_type std     {vt.float().std().item():.4f}   (init 1.0000)")
print(f"\n{'':18s} {'content fraction':>18s} {'weather/metadata':>18s}")
print(f"  fresh init       {cf_fresh:17.2f}% {r_fresh:18.3f}")
if cf_train is not None:
    print(f"  trained          {cf_train:17.2f}% {r_train:18.3f}")
print(f"  after fix        {cf_fixed:17.2f}% {r_fixed:18.3f}")
print("\nParity for five summed branches = 20.00% content fraction, ratio ~1.0.")
print("Floor = 5%.  Pre-rebalance this read 0.04% and 0.04.")
print("="*68)

v15 TOKEN BALANCE — SUMMARY
checkpoint       best.ckpt   epoch 61
var_weights std  0.2757   (init 0.0716 -> 3.9x)
var_type std     1.0164   (init 1.0000)

                     content fraction   weather/metadata
  fresh init                   14.22%              0.764
  trained                       2.83%              0.374


NameError: name 'cf_fixed' is not defined